# 1. Two Sum

**Easy**

Given an array of integers `nums` and an integer `target`, return *indices of the two numbers such that they add up to `target`*.

You may assume that each input would have **exactly one solution**, and you may not use the *same* element twice.

You can return the answer in any order.

---

**Example 1:**

```
Input:  nums = [2, 7, 11, 15], target = 9
Output: [0, 1]
```

**Example 2:**

```
Input:  nums = [3, 2, 4], target = 6
Output: [1, 2]
```

**Example 3:**

```
Input:  nums = [3, 3], target = 6
Output: [0, 1]
```

---

**Constraints:**

- `2 <= nums.length <= 10^4`
- `-10^9 <= nums[i] <= 10^9`
- `-10^9 <= target <= 10^9`
- Only one valid answer exists.

**Follow up:** Can you come up with an algorithm that is less than `O(n^2)` time complexity?  ->  **yes, V4 (hash map) below.**


## Helper : a small Binary Search Tree

Used only by **V3**. Each node stores a value *and* the original index it came
from, so sorting-into-a-tree never loses positions. Equal values are pushed to
the right. Run this cell before the `Solution` cell.

In [2]:
class Tree:
    def __init__(self, val: int = 0, indx: int = 0, left=None, right=None):
        self.val = val
        self.indx = indx
        self.left = left
        self.right = right

    def appendOne(self, val: int, indx: int):
        current = self
        while True:
            if val < current.val:
                if current.left is None:
                    current.left = Tree(val, indx)
                    return
                current = current.left
            else:
                if current.right is None:
                    current.right = Tree(val, indx)
                    return
                current = current.right

    def appendAll(self, lst: list[int]):
        if not lst:
            return
        self.val = lst[0]
        self.indx = 0
        for i in range(1, len(lst)):
            self.appendOne(lst[i], i)

    def getByValueAndIndex(self, val: int, indx: int):
        current = self
        while current:
            if val < current.val:
                current = current.left
            elif val > current.val:
                current = current.right
            else:                          # same value
                if current.indx != indx:   # ... but a different element
                    return current
                current = current.right    # skip myself, look further right
        return None


In [3]:
class Solution:

    # ============ V1 : brute force ============
    # try every pair (i, j) with j > i so an element is never used twice.
    # time O(n^2) | space O(1)
    def twoSumBrute(self, nums: list[int], target: int) -> list[int]:
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                if nums[i] + nums[j] == target:
                    return [i, j]
        return []

    # ============ V2 : complement + membership test ============
    # right idea, wrong container: `in` and `.index` both scan the list,
    # so this is still O(n^2). Kept to show the idea before the hash map.
    # time O(n^2) | space O(1)
    def twoSumIn(self, nums: list[int], target: int) -> list[int]:
        for i, value in enumerate(nums):
            complement = target - value
            if complement in nums:
                j = nums.index(complement)
                if j != i:
                    return [i, j]
        return []

    # ============ V3 : binary search tree ============
    # store every value (with its index) in a BST, then look up each complement.
    # time O(n log n) average, O(n^2) worst (sorted/duplicate input) | space O(n)
    def twoSumTree(self, nums: list[int], target: int) -> list[int]:
        root = Tree()
        root.appendAll(nums)
        for index, value in enumerate(nums):
            node = root.getByValueAndIndex(target - value, index)
            if node:
                return [index, node.indx]
        return []

    # ============ V4 : hash map, one pass  (the intended answer) ============
    # walk once. for each value, ask the dict for its complement BEFORE storing
    # the value -> an element can never pair with itself, and duplicates work.
    # time O(n) | space O(n)
    def twoSumHash(self, nums: list[int], target: int) -> list[int]:
        seen = {}                      # value -> index
        for i, value in enumerate(nums):
            need = target - value
            if need in seen:
                return [seen[need], i]
            seen[value] = i
        return []


In [4]:
# test all four versions against the same cases
sol = Solution()
versions = {
    "V1 brute": sol.twoSumBrute,
    "V2 in":    sol.twoSumIn,
    "V3 tree":  sol.twoSumTree,
    "V4 hash":  sol.twoSumHash,
}

cases = [
    ([2, 7, 11, 15], 9),   # -> {0,1}
    ([3, 2, 4], 6),        # -> {1,2}
    ([3, 3], 6),           # -> {0,1}  (duplicates)
    ([0, 4, 3, 0], 0),     # -> {0,3}  (duplicates + zeros)
    ([-1, -2, -3, -4], -6),# -> {1,3}  (negatives)
]

for name, fn in versions.items():
    line = []
    for nums, target in cases:
        got = fn(list(nums), target)
        ok = len(got) == 2 and nums[got[0]] + nums[got[1]] == target and got[0] != got[1]
        line.append("OK" if ok else f"XX{got}")
    print(f"{name:<9} {line}")


V1 brute  ['OK', 'OK', 'OK', 'OK', 'OK']
V2 in     ['OK', 'OK', 'OK', 'OK', 'OK']
V3 tree   ['OK', 'OK', 'OK', 'OK', 'OK']
V4 hash   ['OK', 'OK', 'OK', 'OK', 'OK']


## Benchmark : the four versions side by side

All four return a correct answer. What separates them is how the running time
grows as the input `n` gets bigger.

| Version | Idea | Time | Space | Verdict |
|---|---|---|---|---|
| **V1 brute** | check every pair | `O(n^2)` | `O(1)` | simple, but dies on big `n` |
| **V2 in** | complement, found with `in` / `.index` | `O(n^2)` | `O(1)` | *feels* smart, secretly still `n^2` |
| **V3 tree** | store values in a BST, look up complements | `O(n log n)` avg, `O(n^2)` worst | `O(n)` | good idea, needs a *balanced* tree to be safe |
| **V4 hash** | store values in a dict, look up complements | **`O(n)`** | `O(n)` | the intended answer |

Two lessons hiding in this table:

1. **V2 vs V4 are the same idea** — "do I already have the complement?" The only
   difference is the *container* you ask. A `list` answers in `O(n)` (it scans);
   a `dict` answers in `O(1)` (it hashes). Same algorithm, wildly different speed.
2. **V3 shows why hash beats tree here.** A BST lookup is `O(log n)` *only if the
   tree stays balanced*. Feed it sorted or repeated values and it collapses into
   a straight line — back to `O(n)` per lookup, `O(n^2)` overall. A dict never
   has that failure mode.

Run the cell below to see the wall-clock times on your own machine.

In [5]:
import time

def bench(fn, nums, target, repeats=3):
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        fn(list(nums), target)
        best = min(best, time.perf_counter() - start)
    return best * 1000     # milliseconds

sol = Solution()
versions = {
    "V1 brute": sol.twoSumBrute,
    "V2 in":    sol.twoSumIn,
    "V3 tree":  sol.twoSumTree,
    "V4 hash":  sol.twoSumHash,
}

# worst case: the only pair is the LAST two elements, so every version
# is forced to look at (almost) the whole array before it finds the answer.
for n in [500, 1000, 2000, 4000]:
    nums = list(range(1, n + 1))
    target = nums[-1] + nums[-2]        # answer = indices [n-2, n-1]
    print(f"n = {n}")
    for name, fn in versions.items():
        print(f"    {name:<9} {bench(fn, nums, target):8.2f} ms")
    print()


n = 500
    V1 brute      8.55 ms
    V2 in         2.59 ms
    V3 tree      22.92 ms
    V4 hash       0.13 ms

n = 1000
    V1 brute     35.85 ms
    V2 in         8.26 ms
    V3 tree      81.01 ms
    V4 hash       0.15 ms

n = 2000
    V1 brute    148.97 ms
    V2 in        32.36 ms
    V3 tree     332.16 ms
    V4 hash       0.24 ms

n = 4000
    V1 brute    581.85 ms
    V2 in       125.75 ms
    V3 tree    1228.33 ms
    V4 hash       0.48 ms

